In [1]:
import os
import pandas as pd
import requests

In [2]:
# Set up headers with token
API_TOKEN = os.getenv("football-data-token")

if not API_TOKEN:
    raise ValueError("football-data-token environment variable is not set.")

HEADERS = {
    "X-Auth-Token": API_TOKEN
}

In [3]:
# Define API endpoints
# url = "https://api.football-data.org/v4/competitions/PL/teams?season=2024" # squads
# url = "https://api.football-data.org/v4//teams/1044/matches" # matches
# url = "https://api.football-data.org/v4/competitions/PL/standings" # standings
# url = "https://api.football-data.org/v4/competitions/PL/matches?matchday=9" # matches

In [4]:
def send_get_request(url, headers=HEADERS) -> dict:
    """Send GET request

    Args:
        url (str): URL to send GET request
        headers (dict): headers to use for GET request

    Returns:
        dict: request payload
    """
    # Send the GET request
    response = requests.get(url=url, headers=headers)

    # Return processed data
    return response.json()


def get_current_matchday() -> int:
    """Get current matchday number

    Returns:
        int: current matchday number
    """
    url = "https://api.football-data.org/v4/competitions/PL"
    data = send_get_request(url, HEADERS)
    return data["currentSeason"]["currentMatchday"]


def get_standings() -> pd.DataFrame:
    """Get league table standings

    Returns:
        pd.DataFrame: league table standings
    """
    # Define standings endpoint
    url = "https://api.football-data.org/v4/competitions/PL/standings"

    # Send the GET request
    data = send_get_request(url, HEADERS)

    for team in data["standings"][0]["table"]:
        team["team"] = team["team"]["name"]

    standings = pd.DataFrame(data["standings"][0]["table"])
    standings["gf_per_game"] = standings["goalsFor"] / standings["playedGames"]
    standings["ga_per_game"] = standings["goalsAgainst"] / standings["playedGames"]

    return standings

In [5]:
def predict_score(home_team, away_team, standings, verbose=False) -> tuple:
    """Get predicted scores for a single match based on stats in standings table

    Args:
        home_team (str): home team name
        away_team (str): away team name
        standings (pd.DataFrame): league standings table
        verbose (bool): whether to print stats behind predicted score

    Returns:
        tuple: (winning team, home team predicted goals, away team predicted goals)
    """
    # Get home team GF/GA stats
    home_team_gf = float(standings[standings["team"] == home_team]["gf_per_game"].values[0])
    home_team_ga = float(standings[standings["team"] == home_team]["ga_per_game"].values[0])

    # Get away team GF/GA stats
    away_team_gf = float(standings[standings["team"] == away_team]["gf_per_game"].values[0])
    away_team_ga = float(standings[standings["team"] == away_team]["ga_per_game"].values[0])
    
    # Compute expected goals
    home_team_xgoals_raw = round((home_team_gf + away_team_ga) / 2, 2)
    away_team_xgoals_raw = round((away_team_gf + home_team_ga) / 2, 2)
    home_team_xgoals_rounded = round(home_team_xgoals_raw)
    away_team_xgoals_rounded = round(away_team_xgoals_raw)

    if verbose:
        print(f"{home_team} Stats: [{round(home_team_gf, 2)} GF] [{round(home_team_ga, 2)} GA]")
        print(f"{away_team} Stats: [{round(away_team_gf, 2)} GF] [{round(away_team_ga, 2)} GA]")
        print(f"Predicted Result: {home_team} [{home_team_xgoals_raw} - {away_team_xgoals_raw}] {away_team}")
        print()

    winner = "DRAW"
    if home_team_xgoals_rounded > away_team_xgoals_rounded:
        winner = "HOME_TEAM"
    if away_team_xgoals_rounded > home_team_xgoals_rounded:
        winner = "AWAY_TEAM"

    return (winner, home_team_xgoals_rounded, away_team_xgoals_rounded)


def get_matchday_predictions(num, standings, verbose=False) -> pd.DataFrame:
    """Get predicted scores for all matches in a matchday

    Args:
        num (int): matchday number
        standings (pd.DataFrame): league standings table

    Returns:
        pd.DataFrame: DataFrame of match predictions
    """
    # Get data from matchday endpoint
    url = f"https://api.football-data.org/v4/competitions/PL/matches?matchday={num}"
    data = send_get_request(url, HEADERS)

    # Predict results for each match
    results = []
    for m in data["matches"]:
        w, hxg, axg = predict_score(m["homeTeam"]["name"], m["awayTeam"]["name"], standings, verbose=verbose)
        results.append(
            {
                "home": m["homeTeam"]["name"],
                "away": m["awayTeam"]["name"],
                "winner": w,
                "home_goals": hxg,
                "away_goals": axg,

            }
        )

    return pd.DataFrame(results)

In [ ]:
MATCHDAY = get_current_matchday()
print(f"It's matchday {MATCHDAY}!")

STANDINGS = get_standings()
STANDINGS

In [ ]:
preds_df = get_matchday_predictions(MATCHDAY, STANDINGS, verbose=True)
preds_df

In [8]:
# Write predictions to CSV
preds_df.to_csv(f"./predictions/matchday_{MATCHDAY}_preds.csv", index=False)

In [ ]:
for i in range(preds_df.shape[0]):
    predicted_score = f'{preds_df.loc[i, "home"]} [{preds_df.loc[i, "home_goals"]} - {preds_df.loc[i, "away_goals"]}] {preds_df.loc[i, "away"]}'
    print(f"Prediction: {predicted_score}")
    print()

In [ ]:
url = "https://api.football-data.org/v4/competitions/PL/teams?season=2024" # squads

In [ ]:
data = send_get_request(url)

In [ ]:
teams = data["teams"]

In [51]:
SQUADS = {}

for i, team in enumerate(teams):
    pdf = pd.DataFrame(team["squad"])
    pdf["team_id"] = team["id"]
    SQUADS[team["name"]] = pdf

In [ ]:
SQUADS["AFC Bournemouth"]

In [ ]:
FEATURES = [
    
]